# Лаба 1: Линейная регрессия
Запуск модели на California Housing c регуляризацией.

In [1]:

import sys, os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('..')
os.environ.setdefault('MPLCONFIGDIR', 'outputs/mpl_cache')
os.makedirs('outputs/mpl_cache', exist_ok=True)

from src.data.preprocessing import preprocess_california_data
from src.models.linear_regression import LinearRegression
from src.utils.visualization import plot_training_curves

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)
np.random.seed(42)


In [2]:

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

X_train, X_test, y_train, y_test, feature_names = preprocess_california_data('../datasets/California_Houses.csv', device=device)
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"Features: {X_train.shape[1]}")
print(f"Train samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"y stats -> min {y_train.min().item():.1f}, max {y_train.max().item():.1f}, mean {y_train.mean().item():.1f}")


Device: cpu
X_train shape: torch.Size([16512, 13])
X_test shape: torch.Size([4128, 13])
y_train shape: torch.Size([16512])
y_test shape: torch.Size([4128])
Features: 13
Train samples: 16512
Test samples: 4128
y stats -> min 14999.0, max 500001.0, mean 207015.9


In [3]:

learning_rate = 0.05
max_epochs = 300

model = LinearRegression(n_features=X_train.shape[1], lr=learning_rate)
print(f"Learning rate: {learning_rate}")
print(f"Max epochs: {max_epochs}")

model.fit(X_train, y_train.view(-1,1), X_test, y_test.view(-1,1), epochs=max_epochs, print_every=50)
print(f"Epochs: {len(model.history['train_mse'])}")

history = {
    'train_mse': model.history['train_mse'],
    'val_mse': model.history.get('val_mse', [])
}
plot_training_curves(history, title='Linear Regression Training Curves (MSE)', model_type='regression', save_path='../outputs/linreg/linreg_training_curves.png')


Learning rate: 0.05
Max epochs: 300
[epoch   1] train_mse=56254115840.0 rmse=237179.5 mae=207015.9 | val_mse=45482975232.0 rmse=213267.4 mae=185461.2
[epoch  50] train_mse=5013160448.0 rmse=70803.7 mae=52092.5 | val_mse=4958568448.0 rmse=70417.1 mae=51865.4
[epoch 100] train_mse=4864484352.0 rmse=69745.9 mae=51393.5 | val_mse=4881288704.0 rmse=69866.2 mae=51186.0
[epoch 150] train_mse=4805944832.0 rmse=69324.9 mae=51015.4 | val_mse=4866956288.0 rmse=69763.6 mae=50781.1
[epoch 200] train_mse=4777823744.0 rmse=69121.8 mae=50808.0 | val_mse=4864073728.0 rmse=69742.9 mae=50535.9
[epoch 250] train_mse=4761942016.0 rmse=69006.8 mae=50672.5 | val_mse=4861662720.0 rmse=69725.6 mae=50368.6
[epoch 300] train_mse=4751772672.0 rmse=68933.1 mae=50579.6 | val_mse=4858301952.0 rmse=69701.5 mae=50250.3
Epochs: 300


In [4]:

with torch.no_grad():
    preds = model._forward(X_test)
    mse_val = torch.mean((preds - y_test.view(-1,1))**2).item()
print(f"Test MSE: {mse_val:.1f}")


Test MSE: 4858301952.0
